# AIC 2026 — Evidence Phase 00  
# FROZEN SYSTEM SNAPSHOT + QUERY FREEZE — v0.2.1

## TYPE

**NEW evidence-phase harness on top of the EXISTING frozen AIC 2026 system.**

This notebook **does not rebuild Stage00/03/20/24** and does not create a new
retrieval system.

It consumes the same five frozen parents that the Final Competition Notebook
already used successfully.

---

## REQUIRED KAGGLE INPUTS — EXACT SOURCES

Attach exactly:

1. **`nguyenminhtric/final24`**
   - Source notebook lineage: Stage24 ABC Master Rebuild / final offline closure.
   - Expected build: `aic2026-stage24-abc-master-rebuild-v1.0.3`.
   - Role here: closure authority, active-state/provenance authority, actual-PTS lineage.

2. **`nguyenminhtric/20-note`**
   - Source notebook: `20_SHADOW_EVIDENCE_PACK_TOP2_v1_7_3.ipynb`.
   - Expected build: `aic2026-stage20-shadow-evidence-pack-v1.7.3`.
   - Role here: compact full-corpus EvidencePack lineage and corpus consistency.

3. **`selaninguyen/03notre`**
   - Source notebook: Stage03 dual-GPU official visual index.
   - Expected build: `aic2026-kis-slice1-v0.2.2`.
   - Role here: exact official `CanonicalFrameRefRegistry`, `dataset_snapshot_id`,
     `corpus_id`, visual metadata, 177,321 official FrameRefs.

4. **`selaninguyen/gggggg`**
   - Source notebook: Stage00 `00_prepare_data_models`.
   - Role here: raw-dataset/model lineage cross-check and frozen model/config parent.
   - This notebook does NOT reload the model for inference.

5. **`nguyenminhtric/data-batch-1-aic`**
   - Source: original AIC 2026 Batch1 raw corpus.
   - Role here: verify raw-video universe and bind `video_id → raw video path`.

These are the same five core inputs documented by the old Final Competition
Notebook. `DatasetSnapshot` was already `READY_BY_LINEAGE` from
`final24 + Stage03/20`; this notebook materializes an evidence-phase snapshot
manifest from those existing parents instead of asking the user to invent
`CORPUS_MANIFEST`, `BASE_SAMPLE_TABLE`, `CODE_HASH_MANIFEST`, etc.

---

## QUERY INPUT — OPTIONAL FOR CORPUS FREEZE, REQUIRED FOR FULL EVIDENCE FREEZE

Preferred exact file:

```text
evidence_query_workload.csv
```

or the existing Stage23 real workload:

```text
stage23_query_workload.csv
```

Required columns:

```text
query_id
task_type
query_text
```

Optional GT/proxy columns remain compatible with Stage23:

```text
events_json
quest_rewrite_text
quest_exemplar_ref
gt_video_id
gt_start_pts
gt_end_pts
gt_answer
gt_path_json
```

Resolution order:

1. `AIC2026_EVIDENCE_QUERY_WORKLOAD=/exact/path.csv`
2. bounded search for `evidence_query_workload.csv`
3. bounded search for `stage23_query_workload.csv`

If no real query workload is attached, notebook 00 **does not fabricate synthetic
queries**. It creates `evidence_query_workload_TEMPLATE.csv` and publishes:

```text
CORPUS_SNAPSHOT_FREEZE = PASS
QUERY_MANIFEST_FREEZE  = PENDING_REAL_QUERY_WORKLOAD
FULL_EVIDENCE_FREEZE   = NOT_READY
```

Then populate the template / attach a real query file and rerun notebook 00.

---

## WHAT IS REUSED

```text
Final24 closure
Stage20 EvidencePacks
Stage03 official FrameRefs / visual metadata
Stage00 lineage/model cache identity
raw AIC 2026 corpus
```

## WHAT IS CREATED NEW

Only evidence-phase control artifacts:

```text
corpus_snapshot_manifest.parquet
base_video_split_table.parquet
train_video_manifest.parquet
dev_video_manifest.parquet
test_video_manifest.parquet
query_manifest.csv                     (only if real queries exist)
evidence_query_workload_TEMPLATE.csv   (only if real queries do not exist)
frozen_config.json
parent_lineage_manifest.json
freeze_manifest.json
```

## NEXT

```text
00 PASS corpus snapshot
    ↓
01 AutoShot PTS runtime validation
    ↓
02 BEiT3 shadow
```

Notebook 01/02 corpus-runtime validation may proceed after corpus snapshot freeze. Training/accuracy benchmark must not proceed until the query manifest is real and frozen.

## v0.2.1 — executed-run correction

**Parent executed evidence:** v0.2.0 successfully mounted/verified Final24, Stage20,
Stage03, Stage00 and raw Batch1, and materialized 873 videos / 177,321 official
FrameRefs.

The executed run exposed two implementation issues and one workflow issue:

1. **P1 runtime inefficiency:** v0.2.0 recursively traversed the complete raw dataset
   (`rglob("*")`) only to locate 873 MP4s. The raw dataset also contains ~177k JPGs
   and ~178k JSONs, so corpus materialization took ~22.6 minutes.
   v0.2.1 uses bounded video-root patterns first and only falls back to a pruned walk.

2. **P1 split semantics:** `VIDEO_ID_FAMILY_FALLBACK` produced only 10 groups
   (7 train / 2 dev / 1 test). `L21`, `L22`, ... are storage/video families, not a
   proven semantic source identity. v0.2.1 therefore does **not infer source identity**
   from folder/video prefixes. If no explicit source/program/collection id exists,
   the safe fallback is `VIDEO_ID`, which guarantees no same-video leakage without
   inventing a broader source relation.

3. **Workflow:** a missing query workload blocks accuracy/training evidence, but it
   must **not block corpus-level runtime validation**. Notebook 01/02 can proceed
   after corpus snapshot freeze even while query freeze is pending.

No architecture, Stage03 baseline, Stage20 EvidencePack, Final24 closure, or Online
Master component is changed.

In [1]:
# ── CELL 01: immutable parent/input contract ──────────────────────────────────
from pathlib import Path
from datetime import datetime, timezone
import os, json, hashlib, re
import pandas as pd
import numpy as np

BUILD_VERSION="aic2026-evidence-frozen-system-snapshot-v0.2.1"
STAGE_ID="evidence00_frozen_system_snapshot"

FINAL24_DATASET_HANDLE="nguyenminhtric/final24"
STAGE20_DATASET_HANDLE="nguyenminhtric/20-note"
STAGE03_DATASET_HANDLE="selaninguyen/03notre"
STAGE00_DATASET_HANDLE="selaninguyen/gggggg"
RAW_DATASET_HANDLE="nguyenminhtric/data-batch-1-aic"

EXPECTED_FINAL24_BUILD="aic2026-stage24-abc-master-rebuild-v1.0.3"
EXPECTED_FINAL24_STATUS="PASS_TECHNICAL_STAGE24_ABC_MASTER_REBUILD"
EXPECTED_STAGE20_BUILD="aic2026-stage20-shadow-evidence-pack-v1.7.3"
EXPECTED_STAGE20_ID="20_shadow_evidence_pack"
EXPECTED_STAGE20_HANDOFF_SHA256="c5c204ed16943de07dd92394f563683b8a09e5187770edccdc1dd08e767ce1b7"
EXPECTED_STAGE03_BUILD="aic2026-kis-slice1-v0.2.2"
EXPECTED_STAGE03_ID="03_dualgpu_visual_index"
EXPECTED_STAGE00_ID="00_prepare_data_models"

BASE=Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path.cwd()
WORK_ROOT=BASE/"aic2026_evidence_phase"/"00_frozen_system_snapshot_v021"
WORK_ROOT.mkdir(parents=True,exist_ok=True)

VIDEO_EXTS={".mp4",".mkv",".mov",".avi",".webm",".m4v"}

VALIDATION_STATE={
    "STATIC_CONTRACT_VALIDATION":"PASS",
    "CORPUS_RUNTIME_VALIDATION":"RUN_RESULT_DEPENDENT",
    "ACCURACY_VALIDATION":"NOT_RUN",
    "LATENCY_VALIDATION":"NOT_RUN",
}

In [2]:
# ── CELL 02: bounded Kaggle dataset resolver + hash helpers ───────────────────
def sha256_file(path,chunk=1<<20):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        while True:
            b=f.read(chunk)
            if not b: break
            h.update(b)
    return h.hexdigest()

def canonical_json(obj):
    return json.dumps(obj,sort_keys=True,separators=(",",":"),ensure_ascii=False)

def sha256_json(obj):
    return hashlib.sha256(canonical_json(obj).encode("utf-8")).hexdigest()

def resolve_dataset(handle):
    owner,name=handle.split("/",1)
    candidates=[
        Path("/kaggle/input")/name,
        Path("/kaggle/input/datasets")/owner/name,
        Path("/kaggle/input/notebooks")/owner/name,
    ]
    hits=[p.resolve() for p in candidates if p.is_dir()]
    if not hits:
        raise RuntimeError({
            "message":"REQUIRED_KAGGLE_INPUT_NOT_MOUNTED",
            "handle":handle,
            "candidates":[str(p) for p in candidates],
            "action":f"Kaggle Add Input: {handle}",
        })
    # Prefer explicit /datasets/owner/name when more than one alias exists.
    hits=sorted(set(hits),key=lambda p:(0 if "/datasets/" in str(p) else 1,len(str(p)),str(p)))
    return hits[0]

def unique_under(root,name):
    matches=sorted({p.resolve() for p in Path(root).rglob(name) if p.is_file()},key=str)
    if len(matches)!=1:
        raise RuntimeError(("UNIQUE_ARTIFACT_REQUIRED",name,str(root),[str(x) for x in matches[:20]]))
    return matches[0]

def stage_root_from_handoff(handoff_path):
    # Published stages normally store handoff under export/.
    p=Path(handoff_path).resolve()
    return p.parent.parent if p.parent.name=="export" else p.parent

def rebase(stage_root,stored_path,fallback_name):
    stored=Path(str(stored_path))
    candidates=[]
    if stored.is_file(): candidates.append(stored.resolve())
    if not stored.is_absolute():
        q=Path(stage_root)/stored
        if q.is_file(): candidates.append(q.resolve())
    candidates.extend(p.resolve() for p in Path(stage_root).rglob(fallback_name) if p.is_file())
    candidates=sorted(set(candidates),key=str)
    if len(candidates)!=1:
        raise RuntimeError(("REBASE_AMBIGUOUS_OR_MISSING",fallback_name,[str(x) for x in candidates[:20]]))
    return candidates[0]

FINAL24_DATASET_ROOT=resolve_dataset(FINAL24_DATASET_HANDLE)
STAGE20_DATASET_ROOT=resolve_dataset(STAGE20_DATASET_HANDLE)
STAGE03_DATASET_ROOT=resolve_dataset(STAGE03_DATASET_HANDLE)
STAGE00_DATASET_ROOT=resolve_dataset(STAGE00_DATASET_HANDLE)
RAW_DATASET_ROOT=resolve_dataset(RAW_DATASET_HANDLE)

print("[PASS] exact frozen parents mounted",{
    "final24":str(FINAL24_DATASET_ROOT),
    "stage20":str(STAGE20_DATASET_ROOT),
    "stage03":str(STAGE03_DATASET_ROOT),
    "stage00":str(STAGE00_DATASET_ROOT),
    "raw":str(RAW_DATASET_ROOT),
})

[PASS] exact frozen parents mounted {'final24': '/kaggle/input/datasets/nguyenminhtric/final24', 'stage20': '/kaggle/input/datasets/nguyenminhtric/20-note', 'stage03': '/kaggle/input/datasets/selaninguyen/03notre', 'stage00': '/kaggle/input/datasets/selaninguyen/gggggg', 'raw': '/kaggle/input/datasets/nguyenminhtric/data-batch-1-aic'}


In [3]:
# ── CELL 03: verify Final24 closure authority ─────────────────────────────────
FINAL24_HANDOFF_PATH=unique_under(FINAL24_DATASET_ROOT,"stage24_handoff.json")
FINAL24_ROOT=stage_root_from_handoff(FINAL24_HANDOFF_PATH)
FINAL24_DECISION_PATH=unique_under(FINAL24_ROOT,"stage24_master_decision.json")
FINAL24_MANIFEST_PATH=unique_under(FINAL24_ROOT,"stage24_artifact_manifest.json")

FINAL24_HANDOFF=json.loads(FINAL24_HANDOFF_PATH.read_text(encoding="utf-8"))
FINAL24_DECISION=json.loads(FINAL24_DECISION_PATH.read_text(encoding="utf-8"))
FINAL24_MANIFEST=json.loads(FINAL24_MANIFEST_PATH.read_text(encoding="utf-8"))

if FINAL24_HANDOFF.get("build_version")!=EXPECTED_FINAL24_BUILD:
    raise AssertionError(("FINAL24_BUILD",FINAL24_HANDOFF.get("build_version"),EXPECTED_FINAL24_BUILD))
if FINAL24_DECISION.get("status")!=EXPECTED_FINAL24_STATUS:
    raise AssertionError(("FINAL24_STATUS",FINAL24_DECISION.get("status"),EXPECTED_FINAL24_STATUS))
if FINAL24_DECISION.get("active_state")!={"KIS":"v0","TRAKE":"v3","QA":"v0_PENDING_FINAL_API"}:
    raise AssertionError(("FINAL24_ACTIVE_STATE",FINAL24_DECISION.get("active_state")))
if FINAL24_DECISION.get("probability_calibration_status")!="NOT_CALIBRATED":
    raise AssertionError("FINAL24_CALIBRATION_STATE_DRIFT")

print("[PASS] Final24 closure",{
    "build":FINAL24_HANDOFF.get("build_version"),
    "status":FINAL24_DECISION.get("status"),
    "active_state":FINAL24_DECISION.get("active_state"),
    "handoff_sha256":sha256_file(FINAL24_HANDOFF_PATH),
    "manifest_sha256":sha256_file(FINAL24_MANIFEST_PATH),
})

[PASS] Final24 closure {'build': 'aic2026-stage24-abc-master-rebuild-v1.0.3', 'status': 'PASS_TECHNICAL_STAGE24_ABC_MASTER_REBUILD', 'active_state': {'KIS': 'v0', 'TRAKE': 'v3', 'QA': 'v0_PENDING_FINAL_API'}, 'handoff_sha256': '8c5f4515bbac9bef416ea75f9e03a0e542d6bb24ff45e0650e20979a92419098', 'manifest_sha256': '21704ffe8b98c0da2b735110357ce6f7965606ec9802b917de84c0eb6b1095ca'}


In [4]:
# ── CELL 04: verify Stage03 exact official FrameRef parent ────────────────────
H03_PATH=unique_under(STAGE03_DATASET_ROOT,"stage03_handoff.json")
H03=json.loads(H03_PATH.read_text(encoding="utf-8"))
if H03.get("stage_id")!=EXPECTED_STAGE03_ID or H03.get("build_version")!=EXPECTED_STAGE03_BUILD:
    raise AssertionError(("STAGE03_IDENTITY",H03.get("stage_id"),H03.get("build_version")))

S03=stage_root_from_handoff(H03_PATH)
VIS_META_PATH=rebase(S03,H03.get("visual_metadata"),"visual_metadata.parquet")
VIS_INDEX_MANIFEST_PATH=rebase(S03,H03.get("visual_index_manifest"),"visual_index_manifest.json")
VIS_INDEX_MANIFEST=json.loads(VIS_INDEX_MANIFEST_PATH.read_text(encoding="utf-8"))

META=pd.read_parquet(VIS_META_PATH)
if "frame_ref_id" not in META.columns or "video_id" not in META.columns:
    raise RuntimeError(("STAGE03_METADATA_SCHEMA",META.columns.tolist()))
if META.frame_ref_id.astype(str).duplicated().any():
    raise RuntimeError("DUPLICATE_STAGE03_FRAME_REF_ID")
if len(META)!=177_321:
    raise RuntimeError(("STAGE03_OFFICIAL_ROW_COUNT_DRIFT",len(META),177321))

DATASET_SNAPSHOT_ID=str(H03.get("dataset_snapshot_id"))
CORPUS_ID=str(H03.get("corpus_id"))
FRAME_REGISTRY_SHA256=str(H03.get("frame_registry_sha256"))

print("[PASS] Stage03 exact official registry",{
    "rows":len(META),
    "dataset_snapshot_id":DATASET_SNAPSHOT_ID,
    "corpus_id":CORPUS_ID,
    "frame_registry_sha256":FRAME_REGISTRY_SHA256,
    "visual_metadata_sha256":sha256_file(VIS_META_PATH),
})

[PASS] Stage03 exact official registry {'rows': 177321, 'dataset_snapshot_id': 'sha256:f9480c86372388b93a8c801357974adda22899846bb65f923ceb6c415b23cca5', 'corpus_id': 'aic2026_batch1', 'frame_registry_sha256': '2186aa8f53a98a09063ec03b155a4fda93f15e2effbd583dd6cc45923ca04695', 'visual_metadata_sha256': 'ab692d3588ebbf5b1aa5ec9010169e24f0fd42d70e1960bc59448c6e9e9d9a7a'}


In [5]:
# ── CELL 05: verify Stage20 compact evidence lineage ──────────────────────────
H20_PATH=unique_under(STAGE20_DATASET_ROOT,"stage20_handoff.json")
H20=json.loads(H20_PATH.read_text(encoding="utf-8"))
if H20.get("stage_id")!=EXPECTED_STAGE20_ID or H20.get("build_version")!=EXPECTED_STAGE20_BUILD:
    raise AssertionError(("STAGE20_IDENTITY",H20.get("stage_id"),H20.get("build_version")))
actual_h20_sha=sha256_file(H20_PATH)
if actual_h20_sha!=EXPECTED_STAGE20_HANDOFF_SHA256:
    raise AssertionError(("STAGE20_HANDOFF_SHA",actual_h20_sha,EXPECTED_STAGE20_HANDOFF_SHA256))

PACK_PATH=rebase(stage_root_from_handoff(H20_PATH),H20.get("evidence_pack_registry_parquet"),"shadow_evidence_pack_registry.parquet")
PACK=pd.read_parquet(PACK_PATH)
if len(PACK)!=18_660:
    raise RuntimeError(("STAGE20_PACK_COUNT_DRIFT",len(PACK),18660))

print("[PASS] Stage20 compact evidence",{
    "packs":len(PACK),
    "handoff_sha256":actual_h20_sha,
    "pack_sha256":sha256_file(PACK_PATH),
})

[PASS] Stage20 compact evidence {'packs': 18660, 'handoff_sha256': 'c5c204ed16943de07dd92394f563683b8a09e5187770edccdc1dd08e767ce1b7', 'pack_sha256': '4246f43f329e34dcbfd0dd39e968ca84c70c7e8c5e96783a635d44eeb533f6d1'}


In [6]:
# ── CELL 06: verify Stage00 lineage against raw corpus ────────────────────────
H00_PATH=unique_under(STAGE00_DATASET_ROOT,"stage00_handoff.json")
H00=json.loads(H00_PATH.read_text(encoding="utf-8"))
if H00.get("stage_id")!=EXPECTED_STAGE00_ID:
    raise AssertionError(("STAGE00_ID",H00.get("stage_id")))
if H00.get("dataset_handle")!=RAW_DATASET_HANDLE:
    raise AssertionError(("STAGE00_RAW_DATASET",H00.get("dataset_handle"),RAW_DATASET_HANDLE))
if int(H00.get("required_gpu_count",-1))!=2:
    raise AssertionError("STAGE00_STRICT_TWO_GPU_DRIFT")
if H00.get("allow_single_gpu_fallback") is not False or H00.get("allow_cpu_model_fallback") is not False:
    raise AssertionError("STAGE00_FALLBACK_POLICY_DRIFT")

print("[PASS] Stage00 lineage",{
    "build":H00.get("build_version"),
    "raw_dataset_handle":H00.get("dataset_handle"),
    "handoff_sha256":sha256_file(H00_PATH),
})

[PASS] Stage00 lineage {'build': 'aic2026-rebuild-batch01-v0.1.2', 'raw_dataset_handle': 'nguyenminhtric/data-batch-1-aic', 'handoff_sha256': '7c4e04fb95fcd933b49c2474eb597170824e52ad71531e6f23a5b994c4567d0c'}


In [7]:
# ── CELL 07: FAST materialization of corpus snapshot from frozen lineage ──────
import time
t_snapshot0=time.perf_counter()

# Stage03 is the exact official video universe authority.
official_video_ids=sorted(META.video_id.astype(str).unique().tolist())
official_video_id_set=set(official_video_ids)
if len(official_video_ids)!=873:
    raise RuntimeError(("STAGE03_VIDEO_UNIVERSE_DRIFT",len(official_video_ids),873))

def bounded_raw_video_candidates(root):
    """
    Prefer known AIC Batch1 video-root structure without recursively walking the
    350k+ auxiliary files. Patterns are storage discovery only, never split semantics.
    """
    root=Path(root)
    patterns=(
        "Videos_*/video/*.mp4",
        "Videos_*/video/*.mkv",
        "Videos_*/*/*.mp4",
        "Videos_*/*/*.mkv",
    )
    hits=[]
    for pat in patterns:
        hits.extend(p.resolve() for p in root.glob(pat) if p.is_file())
    # Deduplicate aliases/overlapping patterns.
    return sorted(set(hits),key=str)

def pruned_video_walk(root):
    """
    Last-resort fallback. Prune known non-video component trees before descending.
    This avoids repeating the v0.2.0 full-dataset traversal.
    """
    import os
    root=Path(root)
    prune_tokens=(
        "keyframe","objects","object","map","mapping","metadata","media-info",
        "media_info","clip","feature","embedding",
    )
    out=[]
    for dirpath,dirnames,filenames in os.walk(root):
        dirnames[:]=[
            d for d in dirnames
            if not any(tok in d.lower() for tok in prune_tokens)
        ]
        for name in filenames:
            p=Path(dirpath)/name
            if p.suffix.lower() in VIDEO_EXTS:
                out.append(p.resolve())
    return sorted(set(out),key=str)

raw_videos=bounded_raw_video_candidates(RAW_DATASET_ROOT)
discovery_policy="BOUNDED_VIDEO_ROOT_GLOBS"

# Require exact reconciliation. If bounded roots do not cover the authoritative
# Stage03 video universe, use pruned fallback rather than silently accepting partial data.
bounded_ids={p.stem for p in raw_videos}
if bounded_ids!=official_video_id_set:
    raw_videos=pruned_video_walk(RAW_DATASET_ROOT)
    discovery_policy="PRUNED_VIDEO_ONLY_WALK_FALLBACK"

raw_rows=[]
for p in raw_videos:
    rel=p.relative_to(RAW_DATASET_ROOT).as_posix()
    raw_rows.append({
        "video_id":p.stem,
        "raw_video_path":str(p),
        "raw_relpath":rel,
        "raw_parent":p.parent.name,
        "raw_storage_family":p.parent.parent.name if len(p.parents)>=2 else p.parent.name,
        "raw_size_bytes":int(p.stat().st_size),
    })
RAW_VIDEO=pd.DataFrame(raw_rows)

if RAW_VIDEO.video_id.astype(str).duplicated().any():
    dup=RAW_VIDEO[RAW_VIDEO.video_id.astype(str).duplicated(keep=False)].video_id.astype(str).unique().tolist()
    raise RuntimeError(("RAW_VIDEO_ID_COLLISION",dup[:50]))

raw_id_set=set(RAW_VIDEO.video_id.astype(str))
if raw_id_set!=official_video_id_set:
    missing=sorted(official_video_id_set-raw_id_set)
    extra=sorted(raw_id_set-official_video_id_set)
    raise RuntimeError(("STAGE03_RAW_VIDEO_UNIVERSE_MISMATCH",{
        "missing_raw_for_stage03":missing[:50],
        "extra_raw_not_in_stage03":extra[:50],
        "stage03_videos":len(official_video_id_set),
        "raw_videos":len(raw_id_set),
        "discovery_policy":discovery_policy,
    }))

meta_cols=set(META.columns)
agg_spec={"official_frame_count":("frame_ref_id","count")}
if "pts_time" in meta_cols:
    agg_spec["official_pts_min"]=("pts_time","min")
    agg_spec["official_pts_max"]=("pts_time","max")
elif "pts_time_technical" in meta_cols:
    agg_spec["official_pts_min"]=("pts_time_technical","min")
    agg_spec["official_pts_max"]=("pts_time_technical","max")

VIDEO_OFFICIAL=META.groupby("video_id",as_index=False).agg(**agg_spec)
VIDEO_OFFICIAL["video_id"]=VIDEO_OFFICIAL.video_id.astype(str)
RAW_VIDEO["video_id"]=RAW_VIDEO.video_id.astype(str)

SNAP=VIDEO_OFFICIAL.merge(RAW_VIDEO,on="video_id",how="inner",validate="one_to_one")

# Split-group policy:
# 1) use explicit semantic source identity only when the frozen parent actually has it;
# 2) otherwise use video_id. Storage folder and Lxx prefix are NOT treated as semantic source.
candidate_group_cols=[
    c for c in ("source_id","program_id","collection_id")
    if c in META.columns
]
if candidate_group_cols:
    gcol=candidate_group_cols[0]
    pair=META[["video_id",gcol]].dropna().drop_duplicates()
    per_video=pair.groupby("video_id")[gcol].nunique()
    if (per_video>1).any():
        raise RuntimeError(("AMBIGUOUS_EXPLICIT_SOURCE_GROUP",gcol,int((per_video>1).sum())))
    gmap=pair.drop_duplicates("video_id").copy()
    gmap["split_group"]=gmap[gcol].astype(str)
    SNAP=SNAP.merge(gmap[["video_id","split_group"]],on="video_id",how="left")
    if SNAP.split_group.isna().any():
        raise RuntimeError(("EXPLICIT_SOURCE_GROUP_MISSING",gcol,int(SNAP.split_group.isna().sum())))
    split_group_source=f"STAGE03_EXPLICIT_{gcol}"
    source_group_status="KNOWN_EXPLICIT"
else:
    SNAP["split_group"]=SNAP.video_id.astype(str)
    split_group_source="VIDEO_ID"
    source_group_status="UNKNOWN_NOT_INFERRED"

SNAP["dataset_snapshot_id"]=DATASET_SNAPSHOT_ID
SNAP["corpus_id"]=CORPUS_ID
SNAP["frame_registry_sha256"]=FRAME_REGISTRY_SHA256
SNAP["snapshot_source"]="EXISTING_FROZEN_SYSTEM_LINEAGE_FINAL24_STAGE03_STAGE20_RAW"
SNAP["raw_discovery_policy"]=discovery_policy
SNAP["source_group_status"]=source_group_status
SNAP=SNAP.sort_values("video_id").reset_index(drop=True)

CORPUS_SNAPSHOT_PATH=WORK_ROOT/"corpus_snapshot_manifest.parquet"
SNAP.to_parquet(CORPUS_SNAPSHOT_PATH,index=False)

snapshot_materialization_sec=float(time.perf_counter()-t_snapshot0)
print("[PASS] evidence corpus snapshot materialized",{
    "videos":len(SNAP),
    "official_frames":int(SNAP.official_frame_count.sum()),
    "raw_discovery_policy":discovery_policy,
    "split_group_source":split_group_source,
    "source_group_status":source_group_status,
    "snapshot_id":DATASET_SNAPSHOT_ID,
    "materialization_sec":snapshot_materialization_sec,
})

[PASS] evidence corpus snapshot materialized {'videos': 873, 'official_frames': 177321, 'raw_discovery_policy': 'BOUNDED_VIDEO_ROOT_GLOBS', 'split_group_source': 'VIDEO_ID', 'source_group_status': 'UNKNOWN_NOT_INFERRED', 'snapshot_id': 'sha256:f9480c86372388b93a8c801357974adda22899846bb65f923ceb6c415b23cca5', 'materialization_sec': 2.428611148000016}


In [8]:
# ── CELL 08: freeze train/dev/test VIDEO manifests before any silver labels ───
def split_for_group(group):
    h=int(hashlib.sha256(str(group).encode()).hexdigest()[:8],16)%100
    return "train" if h<70 else ("dev" if h<85 else "test")

SPLIT=SNAP[[
    "video_id","split_group","raw_relpath","dataset_snapshot_id","corpus_id",
    "source_group_status"
]].copy()
SPLIT["split"]=SPLIT.split_group.astype(str).map(split_for_group)

if (SPLIT.groupby("video_id").split.nunique()>1).any():
    raise RuntimeError("VIDEO_SPLIT_LEAKAGE")
if (SPLIT.groupby("split_group").split.nunique()>1).any():
    raise RuntimeError("GROUP_SPLIT_LEAKAGE")

# The fallback must provide enough independent groups for a meaningful split.
group_counts=SPLIT.groupby("split").split_group.nunique().to_dict()
if split_group_source=="VIDEO_ID" and min(group_counts.get(k,0) for k in ("train","dev","test"))<20:
    raise RuntimeError(("TOO_FEW_VIDEO_LEVEL_SPLIT_GROUPS",group_counts))

BASE_SPLIT_PATH=WORK_ROOT/"base_video_split_table.parquet"
SPLIT.to_parquet(BASE_SPLIT_PATH,index=False)

split_paths={}
for split in ("train","dev","test"):
    p=WORK_ROOT/f"{split}_video_manifest.parquet"
    SPLIT[SPLIT.split.eq(split)].sort_values(["split_group","video_id"]).to_parquet(p,index=False)
    split_paths[split]=p

split_counts=SPLIT.split.value_counts().to_dict()
split_fraction={k:float(v/len(SPLIT)) for k,v in split_counts.items()}

print("[PASS] video split manifests frozen BEFORE labels",{
    "rows":len(SPLIT),
    "split_counts":split_counts,
    "split_fraction":split_fraction,
    "group_counts":group_counts,
    "split_group_source":split_group_source,
    "source_group_status":source_group_status,
    "policy":"SPLIT_BEFORE_SILVER_LABEL_GENERATION",
})

[PASS] video split manifests frozen BEFORE labels {'rows': 873, 'split_counts': {'train': 610, 'test': 134, 'dev': 129}, 'split_fraction': {'train': 0.6987399770904925, 'test': 0.15349369988545247, 'dev': 0.14776632302405499}, 'group_counts': {'dev': 129, 'test': 134, 'train': 610}, 'split_group_source': 'VIDEO_ID', 'source_group_status': 'UNKNOWN_NOT_INFERRED', 'policy': 'SPLIT_BEFORE_SILVER_LABEL_GENERATION'}


In [9]:
# ── CELL 09: resolve or create evidence query workload scaffold ───────────────
QUERY_COLUMNS=[
    "query_id","task_type","query_text","events_json","quest_rewrite_text",
    "quest_exemplar_ref","gt_video_id","gt_start_pts","gt_end_pts",
    "gt_answer","gt_path_json",
]

def resolve_query_workload():
    explicit=os.environ.get("AIC2026_EVIDENCE_QUERY_WORKLOAD","").strip()
    if explicit:
        p=Path(explicit)
        if not p.is_file():
            raise RuntimeError(("EXPLICIT_QUERY_WORKLOAD_MISSING",explicit))
        return p.resolve(),"EXPLICIT_ENV"

    # Bounded exact-name probe only; no recursive full-disk scan.
    roots=[Path("/kaggle/input")]
    for root in roots:
        if not root.is_dir(): continue
        direct=[]
        for name in ("evidence_query_workload.csv","stage23_query_workload.csv"):
            direct.extend(root.glob(f"*/{name}"))
            direct.extend(root.glob(f"datasets/*/*/{name}"))
            direct.extend(root.glob(f"notebooks/*/*/{name}"))
        direct=sorted({p.resolve() for p in direct if p.is_file()},key=str)
        if len(direct)==1:
            return direct[0],"BOUNDED_EXACT_NAME_PROBE"
        if len(direct)>1:
            raise RuntimeError(("AMBIGUOUS_QUERY_WORKLOAD", [str(x) for x in direct]))
    return None,"NOT_FOUND"

query_path,query_source=resolve_query_workload()
QUERY_STATUS=None
QUERY_MANIFEST_PATH=None

if query_path is not None:
    Q=pd.read_csv(query_path)
    required={"query_id","task_type","query_text"}
    miss=required-set(Q.columns)
    if miss:
        raise RuntimeError(("QUERY_SCHEMA_MISSING",sorted(miss),str(query_path)))
    if len(Q)==0:
        raise RuntimeError(("QUERY_WORKLOAD_ZERO_ROWS",str(query_path)))
    Q["task_type"]=Q.task_type.astype(str).str.upper()
    if not set(Q.task_type).issubset({"KIS","TRAKE","QA"}):
        raise RuntimeError(("QUERY_TASK_TYPE_INVALID",sorted(set(Q.task_type))))
    if Q.query_id.astype(str).duplicated().any():
        raise RuntimeError("QUERY_ID_DUPLICATE")
    # Bind query to the already frozen VIDEO split only when GT video is known.
    query_split_status="UNASSIGNED_NO_GT_VIDEO"
    if "gt_video_id" in Q.columns:
        qgt=Q.gt_video_id.astype(str).str.strip()
        known=qgt.ne("") & qgt.ne("nan") & qgt.ne("None")
        if known.any():
            split_map=dict(zip(SPLIT.video_id.astype(str),SPLIT.split.astype(str)))
            unknown_gt=sorted(set(qgt[known])-set(split_map))
            if unknown_gt:
                raise RuntimeError(("QUERY_GT_VIDEO_NOT_IN_FROZEN_CORPUS",unknown_gt[:50]))
            Q["query_split"]=qgt.map(split_map)
            query_split_status="BOUND_BY_GT_VIDEO_WHERE_AVAILABLE"
        else:
            Q["query_split"]=None
    else:
        Q["query_split"]=None

    QUERY_MANIFEST_PATH=WORK_ROOT/"query_manifest.csv"
    Q.to_csv(QUERY_MANIFEST_PATH,index=False)
    QUERY_STATUS="FROZEN_REAL_QUERY_WORKLOAD"
else:
    template=pd.DataFrame(columns=QUERY_COLUMNS)
    QUERY_MANIFEST_PATH=WORK_ROOT/"evidence_query_workload_TEMPLATE.csv"
    template.to_csv(QUERY_MANIFEST_PATH,index=False)
    QUERY_STATUS="PENDING_REAL_QUERY_WORKLOAD"

print("[INFO] query freeze",{
    "status":QUERY_STATUS,
    "source":query_source,
    "path":str(QUERY_MANIFEST_PATH),
})

[INFO] query freeze {'status': 'PENDING_REAL_QUERY_WORKLOAD', 'source': 'NOT_FOUND', 'path': '/kaggle/working/aic2026_evidence_phase/00_frozen_system_snapshot_v021/evidence_query_workload_TEMPLATE.csv'}


In [10]:
# ── CELL 10: publish lineage/config/freeze artifacts ──────────────────────────
PARENT_LINEAGE={
    "final24":{
        "dataset_handle":FINAL24_DATASET_HANDLE,
        "source_notebook_family":"Stage24 ABC Master Rebuild / Final24 closure",
        "build_version":FINAL24_HANDOFF.get("build_version"),
        "handoff_sha256":sha256_file(FINAL24_HANDOFF_PATH),
        "manifest_sha256":sha256_file(FINAL24_MANIFEST_PATH),
    },
    "stage20":{
        "dataset_handle":STAGE20_DATASET_HANDLE,
        "source_notebook":"20_SHADOW_EVIDENCE_PACK_TOP2_v1_7_3.ipynb",
        "build_version":H20.get("build_version"),
        "handoff_sha256":sha256_file(H20_PATH),
    },
    "stage03":{
        "dataset_handle":STAGE03_DATASET_HANDLE,
        "source_notebook_family":"Stage03 dual-GPU official visual index",
        "build_version":H03.get("build_version"),
        "handoff_sha256":sha256_file(H03_PATH),
        "dataset_snapshot_id":DATASET_SNAPSHOT_ID,
        "corpus_id":CORPUS_ID,
        "frame_registry_sha256":FRAME_REGISTRY_SHA256,
        "official_frame_rows":len(META),
    },
    "stage00":{
        "dataset_handle":STAGE00_DATASET_HANDLE,
        "source_notebook_family":"Stage00 00_prepare_data_models",
        "build_version":H00.get("build_version"),
        "handoff_sha256":sha256_file(H00_PATH),
    },
    "raw":{
        "dataset_handle":RAW_DATASET_HANDLE,
        "source":"AIC2026 Batch1 raw corpus",
        "video_rows":len(RAW_VIDEO),
    },
}

FROZEN_CONFIG={
    "build_version":BUILD_VERSION,
    "stage_id":STAGE_ID,
    "architecture_status":"FROZEN",
    "parent_handles":{
        "final24":FINAL24_DATASET_HANDLE,
        "stage20":STAGE20_DATASET_HANDLE,
        "stage03":STAGE03_DATASET_HANDLE,
        "stage00":STAGE00_DATASET_HANDLE,
        "raw":RAW_DATASET_HANDLE,
    },
    "split_policy":"HASH_EXPLICIT_SOURCE_IF_AVAILABLE_ELSE_VIDEO_ID_70_15_15_BEFORE_LABEL_GENERATION",
    "split_group_source":split_group_source,
    "source_group_status":source_group_status,
    "raw_discovery_policy":discovery_policy,
    "query_policy":"REAL_QUERY_ONLY_NO_SYNTHETIC_FALLBACK",
    "strict_two_gpu_project_contract":True,
    "canonical_time_identity":"video_id + pts_time + raw video",
    "probability_calibration_status":"NOT_CALIBRATED",
}

PARENT_PATH=WORK_ROOT/"parent_lineage_manifest.json"
CONFIG_PATH=WORK_ROOT/"frozen_config.json"
PARENT_PATH.write_text(json.dumps(PARENT_LINEAGE,ensure_ascii=False,indent=2),encoding="utf-8")
CONFIG_PATH.write_text(json.dumps(FROZEN_CONFIG,ensure_ascii=False,indent=2),encoding="utf-8")

query_real=QUERY_STATUS=="FROZEN_REAL_QUERY_WORKLOAD"
FREEZE={
    "build_version":BUILD_VERSION,
    "created_at_utc":datetime.now(timezone.utc).isoformat(),
    "architecture_status":"FROZEN",
    "corpus_snapshot_freeze":"PASS",
    "query_manifest_freeze":"PASS" if query_real else "PENDING_REAL_QUERY_WORKLOAD",
    "full_evidence_freeze":"PASS" if query_real else "NOT_READY_QUERY_PENDING",
    "dataset_snapshot_id":DATASET_SNAPSHOT_ID,
    "corpus_id":CORPUS_ID,
    "frame_registry_sha256":FRAME_REGISTRY_SHA256,
    "video_count":len(SNAP),
    "official_frame_count":int(SNAP.official_frame_count.sum()),
    "snapshot_materialization_sec":snapshot_materialization_sec,
    "split_group_source":split_group_source,
    "source_group_status":source_group_status,
    "raw_discovery_policy":discovery_policy,
    "artifacts":{
        "corpus_snapshot_manifest":{
            "path":str(CORPUS_SNAPSHOT_PATH),
            "sha256":sha256_file(CORPUS_SNAPSHOT_PATH),
        },
        "base_video_split_table":{
            "path":str(BASE_SPLIT_PATH),
            "sha256":sha256_file(BASE_SPLIT_PATH),
        },
        "split_manifests":{
            k:{"path":str(p),"sha256":sha256_file(p)}
            for k,p in split_paths.items()
        },
        "query_manifest_or_template":{
            "path":str(QUERY_MANIFEST_PATH),
            "sha256":sha256_file(QUERY_MANIFEST_PATH),
            "status":QUERY_STATUS,
        },
        "parent_lineage_manifest":{
            "path":str(PARENT_PATH),
            "sha256":sha256_file(PARENT_PATH),
        },
        "frozen_config":{
            "path":str(CONFIG_PATH),
            "sha256":sha256_file(CONFIG_PATH),
        },
    },
    "next_actions":{
        "corpus_runtime":"RUN_01_AUTOSHOT_PTS_RUNTIME_VALIDATION",
        "accuracy_training":(
            "QUERY_FREEZE_READY"
            if query_real
            else "ATTACH_REAL_QUERY_WORKLOAD_AND_RERUN_00_BEFORE_ACCURACY_OR_TRAINING"
        ),
    },
    "validation_state":{
        "STATIC_CONTRACT_VALIDATION":"PASS",
        "CORPUS_RUNTIME_VALIDATION":"PASS_PARENT_LINEAGE_AND_SNAPSHOT_MATERIALIZATION",
        "ACCURACY_VALIDATION":"NOT_RUN",
        "LATENCY_VALIDATION":"NOT_RUN",
    },
}
FREEZE_PATH=WORK_ROOT/"freeze_manifest.json"
FREEZE_PATH.write_text(json.dumps(FREEZE,ensure_ascii=False,indent=2),encoding="utf-8")

print(json.dumps(FREEZE,ensure_ascii=False,indent=2))
if not query_real:
    print(
        "\n[STOP BEFORE ACCURACY/TRAINING] Corpus freeze is valid, but a real query "
        "workload is still required. Populate evidence_query_workload_TEMPLATE.csv "
        "or attach stage23_query_workload.csv, then rerun notebook 00."
    )

{
  "build_version": "aic2026-evidence-frozen-system-snapshot-v0.2.1",
  "created_at_utc": "2026-08-15T06:34:35.185901+00:00",
  "architecture_status": "FROZEN",
  "corpus_snapshot_freeze": "PASS",
  "query_manifest_freeze": "PENDING_REAL_QUERY_WORKLOAD",
  "full_evidence_freeze": "NOT_READY_QUERY_PENDING",
  "dataset_snapshot_id": "sha256:f9480c86372388b93a8c801357974adda22899846bb65f923ceb6c415b23cca5",
  "corpus_id": "aic2026_batch1",
  "frame_registry_sha256": "2186aa8f53a98a09063ec03b155a4fda93f15e2effbd583dd6cc45923ca04695",
  "video_count": 873,
  "official_frame_count": 177321,
  "snapshot_materialization_sec": 2.428611148000016,
  "split_group_source": "VIDEO_ID",
  "source_group_status": "UNKNOWN_NOT_INFERRED",
  "raw_discovery_policy": "BOUNDED_VIDEO_ROOT_GLOBS",
  "artifacts": {
    "corpus_snapshot_manifest": {
      "path": "/kaggle/working/aic2026_evidence_phase/00_frozen_system_snapshot_v021/corpus_snapshot_manifest.parquet",
      "sha256": "d18c6e5731bf610e55546efe7ff